<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 16px; padding: 40px; font-family: 'Segoe UI', sans-serif; color: white; margin-bottom: 24px;">
  <div style="font-size: 12px; letter-spacing: 3px; text-transform: uppercase; color: #94a3b8; margin-bottom: 8px;">Network INAE Program Tutorial — Hands On</div>
  <h1 style="margin: 0 0 12px 0; font-size: 2.2em; font-weight: 700;">🔌 Socket Programming in Python</h1>
  <p style="margin: 0; color: #cbd5e1; font-size: 1.05em; line-height: 1.6;">
    Building UDP and TCP client–server applications using Python's <code style='background:#1e40af;padding:2px 8px;border-radius:4px;'>socket</code> module.
    From raw datagrams to reliable byte streams — understanding the API that every networked application uses under the hood.
  </p>
  <hr style="border-color: #334155; margin: 24px 0;">
  <div style="display: flex; gap: 32px; font-size: 0.9em; color: #94a3b8; flex-wrap: wrap;">
    <span>⏱ Estimated Time: ~60 min</span>
    <span>💻 Platform: Linux / macOS / Windows</span>
    <span>🐍 Python 3.8+</span>
    <span>✍️ Author: Mayank</span>
  </div>
</div>

## 📋 Lab Overview

| Section | Topic | What You Do |
|---------|-------|-------------|
| 1 | Reading & Background | Core concepts, references |
| 2 | The `socket` API | Every function you need |
| 3 | UDP — Server & Client | Run both, observe port behaviour |
| 4 | Binding the UDP Client | Fix the ephemeral-port problem |
| 5 | TCP — Server & Client | New functions, reliable delivery |
| 6 | UDP vs TCP — Side-by-Side | Compare the two transport protocols |

**Requirements:** Python 3.8+ standard library only — no `pip install` needed.

---
## 📚 Section 1 — Read First: Background & References

Before writing a single line of code, spend 10–15 minutes with these resources. They will make every API call below much clearer.

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 12px; padding: 24px; color: white; font-family: 'Segoe UI', sans-serif; margin: 16px 0;">
  <h3 style="color:#7dd3fc; margin-top:0;">📖 Recommended Reading</h3>

  <table style="width:100%; border-collapse:collapse; font-size:0.93em;">
    <tr style="border-bottom:1px solid #334155;">
      <th style="text-align:left; padding:10px; color:#94a3b8;">Resource</th>
      <th style="text-align:left; padding:10px; color:#94a3b8;">What to focus on</th>
      <th style="text-align:left; padding:10px; color:#94a3b8;">Link</th>
    </tr>
    <tr style="border-bottom:1px solid #1e293b;">
      <td style="padding:10px; color:#7dd3fc;"><strong>Real Python — Socket Programming Guide</strong><br><span style="color:#94a3b8; font-size:0.88em;">Most complete tutorial — start here</span></td>
      <td style="padding:10px; color:#cbd5e1;">What sockets are, AF_INET vs AF_UNIX, SOCK_STREAM vs SOCK_DGRAM, the client–server lifecycle diagram</td>
      <td style="padding:10px;"><a href="https://realpython.com/python-sockets/" style="color:#38bdf8;">realpython.com/python-sockets</a></td>
    </tr>
    <tr style="border-bottom:1px solid #1e293b;">
      <td style="padding:10px; color:#7dd3fc;"><strong>GeeksforGeeks — Socket Programming in Python</strong><br><span style="color:#94a3b8; font-size:0.88em;">Concise with short runnable examples</span></td>
      <td style="padding:10px; color:#cbd5e1;">TCP echo server/client, UDP echo server/client, socket options</td>
      <td style="padding:10px;"><a href="https://www.geeksforgeeks.org/python/socket-programming-python/" style="color:#38bdf8;">geeksforgeeks.org — socket</a></td>
    </tr>
    <tr>
      <td style="padding:10px; color:#7dd3fc;"><strong>Pythontic — UDP Client–Server Example</strong><br><span style="color:#94a3b8; font-size:0.88em;">Focused UDP reference</span></td>
      <td style="padding:10px; color:#cbd5e1;">UDP-specific API, sendto/recvfrom patterns, no-connection model</td>
      <td style="padding:10px;"><a href="https://pythontic.com/modules/socket/udp-client-server-example" style="color:#38bdf8;">pythontic.com — UDP</a></td>
    </tr>
  </table>

  <p style="color:#fbbf24; font-size:0.9em; margin:16px 0 0 0;">
    💡 <strong>Key idea to internalise before continuing:</strong>
    A <em>socket</em> is a file-like object that represents one endpoint of a bidirectional communication channel.
    The operating system manages the actual sending and receiving of bytes; your program just reads and writes to the socket.
  </p>
</div>

### What is Socket Programming?

Every networked application — a web browser, SSH session, Discord message, or DNS lookup — ultimately sends bytes across the network using **sockets**. A socket is an abstraction provided by the OS that lets your program treat a network connection like a file: open it, read from it, write to it, close it.

Python's `socket` module is a thin wrapper around the POSIX socket API (the same C API used by every language), so understanding it here means you understand how networking works in C, Go, Java, and Rust too.

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-left: 4px solid #2563eb; border-radius: 8px; padding: 16px; margin: 12px 0; font-family: 'Segoe UI', sans-serif;">
<strong style="color:#7dd3fc;">Two dimensions to every socket</strong><br><br>
<strong style="color:#e2e8f0;">Address family</strong> — which addressing scheme:<br>
<ul style="color:#94a3b8; margin: 6px 0 12px 16px;">
  <li><code>AF_INET</code> — IPv4 (the one we use)</li>
  <li><code>AF_INET6</code> — IPv6</li>
  <li><code>AF_UNIX</code> — local inter-process communication (no network at all)</li>
</ul>
<strong style="color:#e2e8f0;">Socket type</strong> — which transport behaviour:<br>
<ul style="color:#94a3b8; margin: 6px 0 0 16px;">
  <li><code>SOCK_DGRAM</code> — UDP: connectionless datagrams, no delivery guarantee</li>
  <li><code>SOCK_STREAM</code> — TCP: connection-oriented, ordered, reliable byte stream</li>
</ul>
</div>

---
## 🔧 Section 2 — The `socket` API: Functions You Need to Know

### 2.1 — Common to Both UDP and TCP

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 12px; padding: 24px; color: white; font-family: 'Segoe UI', sans-serif; margin: 8px 0;">
<h3 style="color:#7dd3fc; margin-top:0;"><code>socket.socket(family, type)</code> — Create a socket</h3>
<p style="color:#cbd5e1;">This is the constructor. It asks the OS to allocate a socket and return a file descriptor for it. Nothing is sent over the network at this point.</p>
<pre style="background:#0f2744; padding:12px; border-radius:6px; color:#e2e8f0; font-size:0.88em;">
import socket
#
#DP socket (AF_INET = IPv4, SOCK_DGRAM = UDP)
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
#
# TCP socket (AF_INET = IPv4, SOCK_STREAM = TCP)
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
</pre>
<p style="color:#94a3b8; font-size:0.88em; margin:8px 0 0 0;">
<strong style="color:#fbbf24;">Default values:</strong> <code>AF_INET</code> and <code>SOCK_STREAM</code> are the defaults — <code>socket.socket()</code> with no arguments creates a TCP/IPv4 socket.
</p>
</div>

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 12px; padding: 24px; color: white; font-family: 'Segoe UI', sans-serif; margin: 8px 0;">
<h3 style="color:#7dd3fc; margin-top:0;"><code>sock.bind((host, port))</code> — Attach to an address</h3>
<p style="color:#cbd5e1;">Binds the socket to a specific IP address and port number. After binding, the OS knows to deliver packets arriving at <code>host:port</code> to this socket.</p>
<pre style="background:#0f2744; padding:12px; border-radius:6px; color:#e2e8f0; font-size:0.88em;">
# Bind to all network interfaces, port 5000
sock.bind(("0.0.0.0", 5000))
#
#Bind to localhost only
sock.bind(("127.0.0.1", 5000))
#
#Bind to a specific interface IP
sock.bind(("192.168.1.10", 5000))
</pre>
<ul style="color:#94a3b8; margin:8px 0 0 0; font-size:0.9em;">
  <li><code>"0.0.0.0"</code> means <em>any interface</em> — the socket accepts traffic on all NICs.</li>
  <li><strong>Servers always bind.</strong> Clients usually don't — the OS assigns an <em>ephemeral port</em> automatically. We will see why this matters in Section 3.</li>
  <li>You cannot bind two sockets to the same port unless you set <code>SO_REUSEADDR</code>.</li>
</ul>
</div>

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 12px; padding: 24px; color: white; font-family: 'Segoe UI', sans-serif; margin: 8px 0;">
<h3 style="color:#7dd3fc; margin-top:0;"><code>sock.sendto(data, (host, port))</code> — Send a datagram  <span style="font-size:0.7em; color:#f59e0b;">UDP only</span></h3>
<p style="color:#cbd5e1;">Sends <code>data</code> (must be <code>bytes</code>) as a single UDP datagram to the specified address. Because UDP is connectionless, you must supply the destination address every time.</p>
<pre style="background:#0f2744; padding:12px; border-radius:6px; color:#e2e8f0; font-size:0.88em;">
message = "Hello Server"
sock.sendto(message.encode(), ("127.0.0.1", 5000))
#                 ^^^^^^^^
#          str → bytes: always encode before sending
</pre>
<p style="color:#94a3b8; font-size:0.88em; margin:8px 0 0 0;">
<strong style="color:#fbbf24;">Important:</strong> <code>sendto()</code> returns immediately even if nobody is listening — UDP has no acknowledgement mechanism. If the server is down, the datagram is silently lost.
</p>
</div>

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 12px; padding: 24px; color: white; font-family: 'Segoe UI', sans-serif; margin: 8px 0;">
<h3 style="color:#7dd3fc; margin-top:0;"><code>sock.recvfrom(bufsize)</code> — Receive a datagram  <span style="font-size:0.7em; color:#f59e0b;">UDP only</span></h3>
<p style="color:#cbd5e1;">Blocks until a datagram arrives, then returns <code>(data, address)</code> — the raw bytes and the <code>(ip, port)</code> tuple of whoever sent it. The <code>bufsize</code> argument sets the maximum number of bytes to read in one call.</p>
<pre style="background:#0f2744; padding:12px; border-radius:6px; color:#e2e8f0; font-size:0.88em;">
data, addr = sock.recvfrom(1024)   # read up to 1024 bytes
message = data.decode()            # bytes → str
print(f"From {addr}: {message}")   # addr = ('ip', port)
</pre>
<ul style="color:#94a3b8; margin:8px 0 0 0; font-size:0.9em;">
  <li>If the datagram is larger than <code>bufsize</code>, the excess bytes are silently discarded (unlike TCP).</li>
  <li>The returned <code>addr</code> tells the server where to send its reply — this is how <code>server_socket.sendto(response, client_addr)</code> works.</li>
</ul>
</div>

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 12px; padding: 24px; color: white; font-family: 'Segoe UI', sans-serif; margin: 8px 0;">
<h3 style="color:#7dd3fc; margin-top:0;"><code>sock.connect((host, port))</code> — Set default remote address</h3>
<p style="color:#cbd5e1;">
For <strong>TCP</strong>: initiates the three-way handshake and establishes the connection. Blocks until the connection is accepted by the server.<br><br>
For <strong>UDP</strong>: does <em>not</em> send anything — it merely records the server's address internally so that you can use <code>sock.send()</code> instead of <code>sock.sendto()</code>. This is called a "connected UDP socket" and is useful when you always talk to the same server.
</p>
<pre style="background:#0f2744; padding:12px; border-radius:6px; color:#e2e8f0; font-size:0.88em;">
# TCP — actual connection attempt:
tcp_sock.connect(("server.example.com", 80))
#
# UDP — just sets default destination (no packet sent):
udp_sock.connect(("127.0.0.1", 5000))
udp_sock.send(b"Hello")   # equivalent to sendto with the stored address
</pre>
</div>

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 12px; padding: 24px; color: white; font-family: 'Segoe UI', sans-serif; margin: 8px 0;">
<h3 style="color:#7dd3fc; margin-top:0;"><code>sock.close()</code> — Release the socket</h3>
<p style="color:#cbd5e1;">Releases the socket and any OS resources associated with it. For TCP, this triggers a FIN and begins the connection teardown sequence. For UDP it simply frees the port.</p>
<pre style="background:#0f2744; padding:12px; border-radius:6px; color:#e2e8f0; font-size:0.88em;">
sock.close()
# Better pattern — use a context manager so close() is called automatically:
with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as sock:
    sock.sendto(b"Hello", ("127.0.0.1", 5000))
    data, _ = sock.recvfrom(1024)
#sock.close() called automatically here
</pre>
<p style="color:#94a3b8; font-size:0.88em; margin:8px 0 0 0;">
Always close sockets when you are done. Leaving them open leaks file descriptors; systems have a finite limit (typically 1024 per process by default).
</p>
</div>

---
## 📡 Section 3 — UDP: Server and Client

### What is UDP?

UDP (User Datagram Protocol) is a **connectionless** transport protocol. There is no handshake, no acknowledgement, no retransmission, no ordering guarantee. A datagram either arrives or it doesn't — the sender never knows which.

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-left: 4px solid #2563eb; border-radius: 8px; padding: 16px; margin: 12px 0;">
<strong style="color:#7dd3fc;">When is UDP the right choice?</strong>
<ul style="color:#94a3b8; margin:8px 0 0 0;">
  <li><strong style="color:#e2e8f0;">DNS</strong> — a 50-byte query and reply; TCP overhead is wasteful</li>
  <li><strong style="color:#e2e8f0;">VoIP / video calls</strong> — a dropped frame is better than a delayed one; retransmitting old audio is pointless</li>
  <li><strong style="color:#e2e8f0;">Online games</strong> — position updates are superseded by the next update anyway</li>
  <li><strong style="color:#e2e8f0;">Streaming media</strong> — slight glitches are preferable to buffering</li>
  <li><strong style="color:#e2e8f0;">Broadcast/multicast</strong> — TCP cannot multicast; UDP can</li>
</ul>
</div>

### UDP Server — Lifecycle

```
socket()  →  bind()  →  recvfrom() [loop]  →  sendto() [loop]
```

The server creates a socket, binds to a well-known port, then loops forever: receive a datagram, send a reply. There is no `listen()` or `accept()` because UDP has no concept of a connection.

### UDP Client — Lifecycle

```
socket()  →  sendto()  →  recvfrom()  →  close()
```

The client needs no `bind()` — the OS picks an available **ephemeral port** (typically 49152–65535) automatically when the first `sendto()` is called.

### 3.1 — UDP Server Code 
##### Do not run the code here

In [ ]:
import socket

# Create UDP socket
server_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

# Bind to IP and port
# "0.0.0.0" means accept on all interfaces (any NIC on this machine)
server_socket.bind(("0.0.0.0", 5000))

print("UDP Server listening on port 5000...")

while True:
    # recvfrom() blocks until a datagram arrives
    # Returns: (bytes, (client_ip, client_port))
    data, client_addr = server_socket.recvfrom(1024)

    message = data.decode()   # bytes → str

    print(f"Received from {client_addr}: {message}")
    # client_addr is a tuple: ('127.0.0.1', <ephemeral_port>)
    # The ephemeral port CHANGES with every new client invocation — see Section 3.3

    # Send response back to whoever sent this datagram
    response = f"Server received: {message}"
    server_socket.sendto(response.encode(), client_addr)
    # We use client_addr from recvfrom() — this is how the server knows where to reply

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-left: 4px solid #2563eb; border-radius: 8px; padding: 16px; margin: 12px 0;">
<strong style="color:#7dd3fc;">📖 Server Code — What each line does</strong>
<ul style="color:#94a3b8; margin:8px 0 0 0;">
  <li><code>SOCK_DGRAM</code> — requests UDP from the OS (not TCP)</li>
  <li><code>bind("0.0.0.0", 5000)</code> — registers this process as the owner of port 5000 on all interfaces; any machine can now reach it at <code>your_ip:5000</code></li>
  <li><code>recvfrom(1024)</code> — sleeps until a datagram arrives; the OS wakes up the process and hands over the bytes + sender address</li>
  <li><code>sendto(response, client_addr)</code> — wraps the reply in a UDP datagram addressed back to the client; no connection state is stored</li>
  <li>The <code>while True</code> loop means the server serves an unlimited number of clients, one message at a time (single-threaded — one client blocks all others while being served)</li>
</ul>
</div>

### 3.2 — UDP Client Code

In [ ]:
import socket

# Create UDP socket — same as server but no bind()
client_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

server_ip   = "127.0.0.1"   # loopback — server is on the same machine
server_port = 5000

messages = [
    "Hello UDP Server",
    "This is my second message to you",
    "Do we have a three way handshake",
    "I guess not, we are UDP",
    "TCP makes a three-way connection",
    "Slow to start, reliable in delivery",
]

# Send all messages one by one and wait for a reply after each
for message in messages:
    # No bind() was called — OS assigns an ephemeral source port here
    client_socket.sendto(message.encode(), (server_ip, server_port))

    # Wait for the server's echo
    response, server_addr = client_socket.recvfrom(1024)
    print("Server Response:", response.decode())

client_socket.close()

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-left: 4px solid #2563eb; border-radius: 8px; padding: 16px; margin: 12px 0;">
<strong style="color:#7dd3fc;">📖 Client Code — What each line does</strong>
<ul style="color:#94a3b8; margin:8px 0 0 0;">
  <li>No <code>bind()</code> — the OS auto-assigns a source port from the ephemeral range (49152–65535) when the first <code>sendto()</code> fires</li>
  <li>No <code>connect()</code> — every <code>sendto()</code> supplies the destination address explicitly</li>
  <li>The client sends a message and immediately waits for a reply — this is a simple synchronous request–response pattern</li>
  <li><code>close()</code> at the end frees the ephemeral port; a new invocation will get a <em>different</em> port number</li>
</ul>
</div>

### 3.3 — Running in the Terminal

<div style="background: #968f01; border-left: 4px solid #eab308; padding: 16px; border-radius: 8px; font-family: monospace; margin: 8px 0;">
<strong>⚡ You need two terminal windows — run these commands in order:</strong>
</div>

**Terminal 1 — Start the server:**
```bash
python3 UDP_server.py
```
You should see:
```
UDP Server listening on port 5000...
```
The server blocks on `recvfrom()`, waiting.

**Terminal 2 — Run the client (while Terminal 1 stays open):**
```bash
python3 UDP_client.py
```

**Terminal 2 output (client):**
```
Server Response: Server received: Hello UDP Server
Server Response: Server received: This is my second message to you
Server Response: Server received: Do we have a three way handshake
...
```

**Terminal 1 output (server):**
```
Received from ('127.0.0.1', 54821): Hello UDP Server
Received from ('127.0.0.1', 54821): This is my second message to you
...
```

**Run the client a second time from Terminal 2:**
```bash
python3 UDP_client.py
```

**Terminal 1 now shows:**
```
Received from ('127.0.0.1', 54821): Hello UDP Server   ← first run
...
Received from ('127.0.0.1', 57103): Hello UDP Server   ← second run — DIFFERENT PORT!
```

### 3.4 — Observation: Every Client Run Uses a Different Port

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-left: 4px solid #f59e0b; border-radius: 8px; padding: 16px; margin: 12px 0; font-family: 'Segoe UI', sans-serif;">
<strong style="color:#fbbf24;">🔍 What you will notice on the server:</strong>
<p style="color:#cbd5e1; margin:8px 0;">
Each time you restart <code>UDP_client.py</code>, the server prints a <strong>different source port</strong> for the client — even though the client IP is the same <code>127.0.0.1</code>. For example:
</p>
<pre style="background:#0f2744; padding:10px; border-radius:6px; color:#e2e8f0; font-size:0.88em;">
Received from ('127.0.0.1', 54821): Hello UDP Server    ← run 1
Received from ('127.0.0.1', 57103): Hello UDP Server    ← run 2
Received from ('127.0.0.1', 49882): Hello UDP Server    ← run 3
</pre>
<p style="color:#cbd5e1; margin:8px 0 0 0;">
<strong>Why?</strong> The client never calls <code>bind()</code>. So each time it starts, the OS assigns a fresh ephemeral port from the available pool (typically 49152–65535 on Linux, or 1024–65535 on some systems). When the old socket is closed, that port is returned to the pool — but the OS does not reuse ports immediately to avoid confusion with delayed packets from the previous session (the <strong>TIME_WAIT</strong> / quiet-time mechanism).
</p>
<p style="color:#fbbf24; font-size:0.9em; margin:12px 0 0 0;">
💡 This is fine for clients — they don't need a predictable port. But if the server changed ports between runs, clients wouldn't know where to connect. That's exactly why <strong>servers always bind to a fixed, well-known port</strong>.
</p>
</div>

---
## 🔒 Section 4 — Binding the UDP Client

Since different client runs show different source ports (as you observed above), what happens if you need the client to always appear from the **same port**? You call `bind()` on the client too.

This is useful in:
- Firewall rules that allow only a specific `client_ip:port` pair
- Protocols where the server needs to identify the client by a fixed port
- Testing and debugging (predictable port makes captures easier to filter)

### 4.1 — UDP Client with Fixed Source Port

In [ ]:
import socket

# Create UDP socket
client_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

# ── Bind the CLIENT to a fixed port ────────────────────────────────────────
# Now this client will always send from port 6000 (on all interfaces)
client_socket.bind(("0.0.0.0", 6000))
print("Client bound to port 6000")
# If you run two instances simultaneously, the second will fail with
# "Address already in use" — a port can only be owned by one socket at a time.

server_ip   = "127.0.0.1"
server_port = 5000

messages = [
    "Hello UDP Server",
    "This is my second message to you",
    "Do we have a three way handshake",
    "I guess not, we are UDP",
    "TCP makes a three-way connection",
    "Slow to start, reliable in delivery",
]

for message in messages:
    client_socket.sendto(message.encode(), (server_ip, server_port))

    response, server_addr = client_socket.recvfrom(1024)
    print("Server Response:", response.decode())

client_socket.close()

**Run the server first, then run this client twice:**

```bash
# Terminal 1
python3 UDP_server.py

# Terminal 2 — run twice
python3 UDP_client_bound.py
python3 UDP_client_bound.py
```

**Expected server output — same port both times:**
```
Received from ('127.0.0.1', 6000): Hello UDP Server   ← run 1
...
Received from ('127.0.0.1', 6000): Hello UDP Server   ← run 2
```

The source port is now **6000** on every run — the client's identity is stable.

<div style="background:linear-gradient(135deg, #0b3e02 0%, #1e5f1f 100%); border-left: 4px solid #16a34a; padding: 16px; border-radius: 8px; margin: 8px 0;">
<strong>❓ Questions — Section 4</strong>
<ol>
<li>What happened when you ran the bound client twice simultaneously? What error did you get and why?</li>
<li>If you want two clients to run at the same time with fixed ports, what would you need to change?</li>
<li>The server uses <code>bind("0.0.0.0", 5000)</code> and the client uses <code>bind("0.0.0.0", 6000)</code>. What would happen if both tried to bind to port 5000?</li>
</ol>
</div>

**✍️ Observations — Section 4:**
```
Client source port (unbound)     : varies — e.g.
Client source port (bound)       : 6000 (fixed)
Error when binding same port twice:
```

---
## 🔗 Section 5 — TCP: Reliable Streams

### What is TCP?

TCP (Transmission Control Protocol) is a **connection-oriented** protocol. Before any data flows, a **three-way handshake** establishes a connection:

```
Client  ──── SYN ──────────────▶  Server
Client  ◀─── SYN-ACK ─────────── Server
Client  ──── ACK ──────────────▶  Server
         (connection established)
```

After the handshake, both sides can send and receive a continuous **byte stream**. TCP guarantees:
- **Delivery** — lost packets are retransmitted
- **Order** — bytes arrive in the order they were sent
- **Flow control** — the receiver can slow down the sender (window)
- **Congestion control** — the sender backs off when the network is congested

### TCP API — New Functions

TCP introduces four functions that have no UDP equivalent:

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 12px; padding: 24px; color: white; font-family: 'Segoe UI', sans-serif; margin: 16px 0 8px 0;">
<table style="width:100%; border-collapse:collapse; font-size:0.91em;">
  <tr style="border-bottom:1px solid #334155;">
    <th style="text-align:left; padding:10px; color:#94a3b8; width:22%;">Function</th>
    <th style="text-align:left; padding:10px; color:#94a3b8;">What it does</th>
    <th style="text-align:left; padding:10px; color:#94a3b8; width:35%;">Signature</th>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#7dd3fc;"><code>listen()</code></td>
    <td style="padding:10px; color:#cbd5e1;">Marks the socket as a <em>passive</em> socket — one that accepts incoming connection requests. Must be called after <code>bind()</code>, before <code>accept()</code>.</td>
    <td style="padding:10px; color:#e2e8f0;"><code>sock.listen(backlog)</code><br><span style="color:#94a3b8; font-size:0.85em;"><code>backlog</code> = max queued connections (e.g. 5)</span></td>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#7dd3fc;"><code>accept()</code></td>
    <td style="padding:10px; color:#cbd5e1;">Blocks until a client connects. Returns a <strong>new socket</strong> for that connection and the client's address. The original listening socket stays open to accept more clients.</td>
    <td style="padding:10px; color:#e2e8f0;"><code>conn, addr = sock.accept()</code><br><span style="color:#94a3b8; font-size:0.85em;"><code>conn</code> is a new connected socket</span></td>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#7dd3fc;"><code>send()</code></td>
    <td style="padding:10px; color:#cbd5e1;">Sends bytes over an established TCP connection. No need to specify an address — the connection already knows the destination. May send fewer bytes than requested; check the return value.</td>
    <td style="padding:10px; color:#e2e8f0;"><code>bytes_sent = conn.send(data)</code></td>
  </tr>
  <tr>
    <td style="padding:10px; color:#7dd3fc;"><code>recv()</code></td>
    <td style="padding:10px; color:#cbd5e1;">Receives up to <code>bufsize</code> bytes from the connection. Blocks until data arrives. Returns an empty <code>b""</code> when the remote side has closed the connection.</td>
    <td style="padding:10px; color:#e2e8f0;"><code>data = conn.recv(bufsize)</code><br><span style="color:#94a3b8; font-size:0.85em;"><code>b""</code> = connection closed</span></td>
  </tr>
</table>
</div>

### 5.1 — TCP Server Code

In [ ]:
import socket

# Create TCP socket (SOCK_STREAM = TCP)
server_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# SO_REUSEADDR lets you restart the server immediately without waiting
# for the OS TIME_WAIT period to expire
server_socket.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)

# Bind to all interfaces on port 5001
# (using 5001 to avoid clashing with UDP server on 5000)
server_socket.bind(("0.0.0.0", 5001))

# listen() — mark as passive; queue up to 5 pending connections
server_socket.listen(5)
print("TCP Server listening on port 5001...")

while True:
    # accept() blocks until a client performs the three-way handshake
    # Returns a *new* socket for this client + the client address
    conn, client_addr = server_socket.accept()
    print(f"\nNew connection from {client_addr}")

    # Handle this client: receive messages until connection closes
    with conn:
        while True:
            # recv() blocks until bytes arrive; returns b"" when client disconnects
            data = conn.recv(1024)
            if not data:
                print(f"Client {client_addr} disconnected.")
                break   # exit inner loop; go back to accept() for next client

            message = data.decode()
            print(f"Received: {message}")

            # send() — no address needed; connection already knows destination
            response = f"Server received: {message}"
            conn.send(response.encode())

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-left: 4px solid #2563eb; border-radius: 8px; padding: 16px; margin: 12px 0;">
<strong style="color:#7dd3fc;">📖 TCP Server — Key differences from UDP Server</strong>
<ul style="color:#94a3b8; margin:8px 0 0 0;">
  <li><code>listen(5)</code> — no UDP equivalent; creates a pending-connection queue of up to 5 SYN packets</li>
  <li><code>accept()</code> — blocks until a full handshake completes; returns a <em>new socket</em> dedicated to this one client. The listening socket stays available for the next caller.</li>
  <li><code>recv()</code> instead of <code>recvfrom()</code> — no address returned because the connection already carries that information</li>
  <li><code>data == b""</code> signals that the client called <code>close()</code> — TCP FIN received</li>
  <li><code>SO_REUSEADDR</code> — prevents "Address already in use" when restarting quickly</li>
</ul>
</div>

### 5.2 — TCP Client Code

In [ ]:
import socket

# Create TCP socket
client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

server_ip   = "127.0.0.1"
server_port = 5001

# connect() initiates the three-way handshake:
#   client → SYN → server
#   client ← SYN-ACK ← server
#   client → ACK → server
# Blocks until the handshake completes (or fails with ConnectionRefusedError)
client_socket.connect((server_ip, server_port))
print(f"Connected to {server_ip}:{server_port}")

messages = [
    "Hello TCP Server",
    "This is my second message to you",
    "We have a three-way handshake here",
    "TCP guarantees delivery and order",
    "If a packet drops, TCP retransmits it",
    "Goodbye — closing the connection now",
]

for message in messages:
    # send() — no address; the connection already knows where to go
    client_socket.send(message.encode())

    # recv() — blocks until the server sends its reply
    response = client_socket.recv(1024)
    print("Server Response:", response.decode())

# close() sends TCP FIN → server's recv() returns b"" → server prints "disconnected"
client_socket.close()
print("Connection closed.")

### 5.3 — Running TCP in the Terminal

**Terminal 1 — Start TCP server:**
```bash
python3 TCP_server.py
```
```
TCP Server listening on port 5001...
```

**Terminal 2 — Run TCP client:**
```bash
python3 TCP_client.py
```

**Terminal 2 (client):**
```
Connected to 127.0.0.1:5001
Server Response: Server received: Hello TCP Server
Server Response: Server received: This is my second message to you
...
Connection closed.
```

**Terminal 1 (server):**
```
New connection from ('127.0.0.1', 52847)
Received: Hello TCP Server
Received: This is my second message to you
...
Client ('127.0.0.1', 52847) disconnected.
```

Notice: the server prints `New connection from ...` **once** — then receives all messages on the **same connection**. With UDP, each `recvfrom()` could come from a completely different address. That is the core difference.

---
## ⚖️ Section 6 — UDP vs TCP: Side-by-Side Comparison

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 12px; padding: 24px; color: white; font-family: 'Segoe UI', sans-serif; margin: 16px 0;">
<h3 style="color:#7dd3fc; margin-top:0;">API Comparison</h3>
<table style="width:100%; border-collapse:collapse; font-size:0.9em;">
  <tr style="border-bottom:1px solid #334155;">
    <th style="text-align:left; padding:10px; color:#94a3b8; width:20%;">Phase</th>
    <th style="text-align:left; padding:10px; color:#3b82f6;">UDP</th>
    <th style="text-align:left; padding:10px; color:#f59e0b;">TCP</th>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#e2e8f0;">Create</td>
    <td style="padding:10px; color:#cbd5e1;"><code>socket(AF_INET, SOCK_DGRAM)</code></td>
    <td style="padding:10px; color:#cbd5e1;"><code>socket(AF_INET, SOCK_STREAM)</code></td>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#e2e8f0;">Server setup</td>
    <td style="padding:10px; color:#cbd5e1;"><code>bind()</code></td>
    <td style="padding:10px; color:#cbd5e1;"><code>bind()</code> → <code>listen()</code></td>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#e2e8f0;">Accept client</td>
    <td style="padding:10px; color:#94a3b8;"><em>not needed — no connection</em></td>
    <td style="padding:10px; color:#cbd5e1;"><code>accept()</code> → new socket per client</td>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#e2e8f0;">Client connect</td>
    <td style="padding:10px; color:#94a3b8;"><em>not needed</em></td>
    <td style="padding:10px; color:#cbd5e1;"><code>connect()</code> — three-way handshake</td>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#e2e8f0;">Send</td>
    <td style="padding:10px; color:#cbd5e1;"><code>sendto(data, addr)</code></td>
    <td style="padding:10px; color:#cbd5e1;"><code>send(data)</code> — no address needed</td>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#e2e8f0;">Receive</td>
    <td style="padding:10px; color:#cbd5e1;"><code>recvfrom(size)</code> → data + addr</td>
    <td style="padding:10px; color:#cbd5e1;"><code>recv(size)</code> → data only</td>
  </tr>
  <tr>
    <td style="padding:10px; color:#e2e8f0;">Close</td>
    <td style="padding:10px; color:#cbd5e1;"><code>close()</code> — frees port</td>
    <td style="padding:10px; color:#cbd5e1;"><code>close()</code> — sends FIN, tears down connection</td>
  </tr>
</table>
</div>

<div style="background: linear-gradient(135deg, #0f172a 0%, #1e3a5f 100%); border-radius: 12px; padding: 24px; color: white; font-family: 'Segoe UI', sans-serif; margin: 16px 0;">
<h3 style="color:#7dd3fc; margin-top:0;">Protocol Comparison</h3>
<table style="width:100%; border-collapse:collapse; font-size:0.9em;">
  <tr style="border-bottom:1px solid #334155;">
    <th style="text-align:left; padding:10px; color:#94a3b8; width:28%;">Property</th>
    <th style="text-align:left; padding:10px; color:#3b82f6;">UDP</th>
    <th style="text-align:left; padding:10px; color:#f59e0b;">TCP</th>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#e2e8f0;">Connection</td>
    <td style="padding:10px; color:#cbd5e1;">Connectionless — no handshake</td>
    <td style="padding:10px; color:#cbd5e1;">Connection-oriented — three-way handshake</td>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#e2e8f0;">Reliability</td>
    <td style="padding:10px; color:#f87171;">No — packets may be lost silently</td>
    <td style="padding:10px; color:#86efac;">Yes — retransmission on loss</td>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#e2e8f0;">Ordering</td>
    <td style="padding:10px; color:#f87171;">No — packets may arrive out of order</td>
    <td style="padding:10px; color:#86efac;">Yes — bytes delivered in order</td>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#e2e8f0;">Header size</td>
    <td style="padding:10px; color:#cbd5e1;">8 bytes</td>
    <td style="padding:10px; color:#cbd5e1;">20–60 bytes</td>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#e2e8f0;">Data unit</td>
    <td style="padding:10px; color:#cbd5e1;">Datagram (discrete message)</td>
    <td style="padding:10px; color:#cbd5e1;">Byte stream (no message boundaries)</td>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#e2e8f0;">Congestion control</td>
    <td style="padding:10px; color:#f87171;">None — sender sets rate freely</td>
    <td style="padding:10px; color:#86efac;">Yes — cwnd, slow start, AIMD</td>
  </tr>
  <tr style="border-bottom:1px solid #1e293b;">
    <td style="padding:10px; color:#e2e8f0;">Broadcast/multicast</td>
    <td style="padding:10px; color:#86efac;">Yes</td>
    <td style="padding:10px; color:#f87171;">No</td>
  </tr>
  <tr>
    <td style="padding:10px; color:#e2e8f0;">Typical uses</td>
    <td style="padding:10px; color:#cbd5e1;">DNS, VoIP, video streaming, gaming</td>
    <td style="padding:10px; color:#cbd5e1;">HTTP/S, SSH, email, file transfer</td>
  </tr>
</table>
</div>

### Lifecycle Diagrams

**UDP (no connection):**
```
SERVER                          CLIENT
socket()                        socket()
bind(5000)                      [no bind — ephemeral port]
recvfrom() ←─── datagram ─────── sendto(5000)
sendto(client) ──── reply ─────▶ recvfrom()
recvfrom() ←─── datagram ─────── sendto(5000)
  ...                              ...
                                close()
```

**TCP (connection-oriented):**
```
SERVER                          CLIENT
socket()                        socket()
bind(5001)
listen()
accept() ◀──── SYN ─────────── connect(5001)
         ────── SYN-ACK ──────▶
accept() ◀──── ACK ────────────  (handshake complete)
recv() ◀────── send() ─────────  send data
send() ─────── recv() ────────▶  receive reply
  ...                              ...
recv() = b"" ◀──── close() ────   (FIN received)
close()
```

<div style="background:linear-gradient(135deg, #0b3e02 0%, #1e5f1f 100%); border-left: 4px solid #16a34a; padding: 16px; border-radius: 8px; margin: 8px 0;">
<strong>❓ Questions — Section 6</strong>
<ol>
<li>Why does TCP need <code>accept()</code> but UDP does not?</li>
<li>In the TCP server, <code>accept()</code> returns a <em>new</em> socket. Why does the server need two sockets — the listening one and the per-client one?</li>
<li>The TCP client calls <code>connect()</code> but the UDP client (unmodified) does not. What does calling <code>connect()</code> on a UDP socket actually do — does it send anything?</li>
<li>What does <code>recv()</code> returning <code>b""</code> mean in TCP? What is the equivalent signal in UDP when the "other side" stops sending?</li>
<li>A UDP server receives a message on port 5000. It replies using <code>sendto(response, client_addr)</code>. The client is unbound — what port does the reply arrive on? How does the client's OS know which application to deliver it to?</li>
<li>If you ran 100 TCP clients simultaneously connecting to the TCP server above, what would happen? (Hint: the server is single-threaded and handles one client at a time.) How would you fix this?</li>
</ol>
</div>

**✍️ Your Observations:**
```
UDP client source port (run 1) :
UDP client source port (run 2) :
UDP client source port (bound) : 6000

TCP — does server print "New connection" once or per message? :
TCP — what does the server print when client calls close()   :

Key difference observed between UDP and TCP server behaviour  :
```

*Analysis:*

---
## ✅ Summary & Key Takeaways

<div style="background:#0f172a; color: #e2e8f0; border-radius: 12px; padding: 24px; font-family: 'Segoe UI', sans-serif; margin: 12px 0;">
<h3 style="color: #7dd3fc; margin-top: 0;">The Socket Programming Toolkit</h3>
<table style="width:100%; border-collapse: collapse;">
  <tr style="border-bottom: 1px solid #334155;">
    <th style="text-align:left; padding: 10px; color: #94a3b8;">Concept</th>
    <th style="text-align:left; padding: 10px; color: #94a3b8;">Key takeaway</th>
  </tr>
  <tr style="border-bottom: 1px solid #1e293b;">
    <td style="padding: 10px; color: #7dd3fc;"><code>socket(AF_INET, SOCK_DGRAM)</code></td>
    <td style="padding: 10px;">UDP — connectionless, low overhead, no guarantees</td>
  </tr>
  <tr style="border-bottom: 1px solid #1e293b;">
    <td style="padding: 10px; color: #7dd3fc;"><code>socket(AF_INET, SOCK_STREAM)</code></td>
    <td style="padding: 10px;">TCP — connection-oriented, ordered, reliable</td>
  </tr>
  <tr style="border-bottom: 1px solid #1e293b;">
    <td style="padding: 10px; color: #7dd3fc;"><code>bind()</code></td>
    <td style="padding: 10px;">Always for servers; optional for clients (fixed port)</td>
  </tr>
  <tr style="border-bottom: 1px solid #1e293b;">
    <td style="padding: 10px; color: #7dd3fc;"><code>listen()</code> + <code>accept()</code></td>
    <td style="padding: 10px;">TCP servers only — queue connections, then handle one by one</td>
  </tr>
  <tr style="border-bottom: 1px solid #1e293b;">
    <td style="padding: 10px; color: #7dd3fc;"><code>sendto()</code> / <code>recvfrom()</code></td>
    <td style="padding: 10px;">UDP — address is per-datagram (no state)</td>
  </tr>
  <tr style="border-bottom: 1px solid #1e293b;">
    <td style="padding: 10px; color: #7dd3fc;"><code>send()</code> / <code>recv()</code></td>
    <td style="padding: 10px;">TCP — no address needed; connection carries it</td>
  </tr>
  <tr>
    <td style="padding: 10px; color: #7dd3fc;">Ephemeral ports</td>
    <td style="padding: 10px;">OS assigns automatically when client omits <code>bind()</code>; changes each run</td>
  </tr>
</table>
<p style="color: #94a3b8; margin: 16px 0 0 0;">The progression: <strong style="color:#7dd3fc;">Create → Bind → (Listen → Accept for TCP) → Send/Receive → Close</strong></p>
</div>

---
## 📎 Appendix — Extended Questions

**A.** Modify the TCP server to handle multiple clients **concurrently** using `threading.Thread`. Each accepted connection should be handled in its own thread. Test with two clients running simultaneously.

**B.** The current TCP server reads up to 1024 bytes per `recv()`. What happens if the client sends a message longer than 1024 bytes? Write a loop that keeps calling `recv()` until it has read a full message (hint: you need a delimiter or a length-prefix framing protocol).

**C.** Set a **timeout** on the UDP client socket using `sock.settimeout(2.0)`. What exception is raised if the server does not reply within 2 seconds? Add try/except to handle it gracefully.

**D.** Use `tcpdump` or Wireshark to capture traffic while running both the UDP and TCP examples, ensure the system has not internet connetion then:
```bash
sudo tcpdump -i lo -n port 5000 or port 5001 -w socket_lab.pcap
```
Open the pcap in Wireshark. Find the TCP three-way handshake (SYN, SYN-ACK, ACK). How many packets does a single TCP message exchange require vs UDP?

**E.** What is `SO_REUSEADDR`? The TCP server sets it — remove it, restart the server quickly after killing it, and observe the error. Why does this happen and what does `SO_REUSEADDR` fix?

---
## 📚 Reference & Further Reading

| Resource | Link |
|----------|------|
| Real Python — Socket Programming Guide | https://realpython.com/python-sockets/ |
| GeeksforGeeks — Socket Programming | https://www.geeksforgeeks.org/python/socket-programming-python/ |
| Pythontic — UDP Client–Server Example | https://pythontic.com/modules/socket/udp-client-server-example |
| Python `socket` module docs | https://docs.python.org/3/library/socket.html |
| RFC 793 — TCP | https://www.rfc-editor.org/rfc/rfc793 |
| RFC 768 — UDP | https://www.rfc-editor.org/rfc/rfc768 |

---
<sub>Notebook authored by <strong>Mayank</strong> with help of Claude.</sub>
<sub> · Network INAE Program · 2026</sub>